# Earth-Moon Transfer Mission

This notebook demonstrates a complete Earth-Moon transfer mission using the Space State simulation framework.

## Mission Profile
1. Start in Low Earth Orbit (LEO) at 400 km altitude
2. Trans-Lunar Injection (TLI) burn
3. Coast to the Moon
4. Lunar Orbit Insertion (LOI) burn
5. Final lunar orbit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../space_state')

from base import (
    Propagator, Spacecraft, Thruster, BODIES,
    get_circular_state, PropagationResult,
    get_body_position, get_body_velocity, SOI_RADII,
)
from base.star_structures import PropagationResult
from base.star_constants import IDX_R, IDX_V, IDX_M, IDX_Q, IDX_OMEGA

## 1. Define Spacecraft

We'll use a spacecraft similar to an Apollo Command/Service Module.

In [ ]:
# Apollo-like spacecraft
main_engine = Thruster(
    name="SPS Engine",
    thrust_n=91000,  # 91 kN (Apollo SPS)
    isp_s=314,       # Specific impulse
)

spacecraft = Spacecraft(
    name="Lunar Transfer Vehicle",
    dry_mass=12000,       # kg (CSM dry mass)
    propellant_mass=18000, # kg (SPS propellant)
    thrusters=[main_engine],
)

print(f"Spacecraft: {spacecraft._name}")
print(f"Total mass: {spacecraft.mass:,.0f} kg")
print(f"Delta-V capacity: {spacecraft.delta_v_remaining:.1f} km/s")

## 2. Initial State in LEO

Set up the spacecraft in a 400 km circular orbit around Earth.

In [ ]:
# Get initial state in LEO
altitude_km = 400
t0 = 0.0  # Start time

# Get circular orbit state (position and velocity relative to Earth)
state = get_circular_state(
    body='earth',
    altitude_km=altitude_km,
    inclination_deg=28.5,  # Kennedy Space Center latitude
    time=t0,
    mass=spacecraft.mass,
)

# Display initial conditions
r_earth = get_body_position('earth', t0)
r_rel = state[:3] - r_earth
v_earth = get_body_velocity('earth', t0)
v_rel = state[3:6] - v_earth

# Convert to km and km/s for display
r_rel_km = np.linalg.norm(r_rel) * 1.496e8
v_rel_kmps = np.linalg.norm(v_rel) * 1731.46

print(f"Initial orbit altitude: {altitude_km} km")
print(f"Distance from Earth center: {r_rel_km:,.0f} km")
print(f"Orbital velocity: {v_rel_kmps:.2f} km/s")
print(f"Orbital period: {2*np.pi*r_rel_km / v_rel_kmps / 60:.1f} minutes")

## 3. Trans-Lunar Injection (TLI)

Calculate the required delta-V for TLI using the vis-viva equation.

In [ ]:
# TLI calculation
# Target: Reach Moon's orbit (~384,400 km) on a Hohmann-like transfer

r_leo_km = 6371 + altitude_km  # Earth radius + altitude
r_moon_km = 384400  # Moon's orbital radius

mu_earth_km = BODIES['earth']['mu_km3_s2']

# Transfer orbit semi-major axis
a_transfer_km = (r_leo_km + r_moon_km) / 2

# Velocities
v_circular_leo = np.sqrt(mu_earth_km / r_leo_km)
v_tli = np.sqrt(mu_earth_km * (2/r_leo_km - 1/a_transfer_km))

delta_v_tli = v_tli - v_circular_leo

print(f"LEO circular velocity: {v_circular_leo:.3f} km/s")
print(f"TLI velocity required: {v_tli:.3f} km/s")
print(f"TLI delta-V: {delta_v_tli:.3f} km/s")

## 4. Apply TLI Burn and Propagate

Set up guidance to apply the TLI burn and propagate to the Moon.

In [ ]:
from guidance.star_tools import BurnGuidance

# Create TLI burn guidance
# Convert delta-V to AU/day
delta_v_au_day = delta_v_tli / 1731.46

# Burn direction is along velocity (prograde)
v_hat = v_rel / np.linalg.norm(v_rel)

tli_guidance = BurnGuidance(
    t_start=t0,
    delta_v=delta_v_au_day * v_hat,  # Prograde TLI burn
    burn_time=None,  # Calculate from delta-V
)

print(f"TLI burn delta-V: {delta_v_tli:.3f} km/s")
print(f"Burn direction (velocity-aligned): {v_hat}")

In [ ]:
# Set up propagator with TLI guidance
result = PropagationResult()
result.initialize(state, t0)

propagator = Propagator(
    ship=spacecraft,
    guidance=tli_guidance,
    result=result,
)

# Propagate through TLI burn and coast to Moon (~3 days)
t_end = 3.0  # days
propagator.propagate(t_end)

print(f"Propagated {len(result.t)} time steps")
print(f"Final time: {result.t[-1]:.2f} days")
print(f"Fuel consumed: {state[6] - result.y[6, -1]:.0f} kg")

## 5. Plot the Transfer Trajectory

In [ ]:
# Plot trajectory in Earth-centered frame
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Get Earth and Moon positions over time
earth_pos = np.array([get_body_position('earth', t) for t in result.t])
moon_pos = np.array([get_body_position('moon', t) for t in result.t])

# Spacecraft relative to Earth
sc_rel_earth = result.y[:3, :].T - earth_pos
moon_rel_earth = moon_pos - earth_pos

# Convert to thousands of km
scale = 1.496e8 / 1000  # AU to 1000 km

# XY plot
ax = axes[0]
ax.plot(sc_rel_earth[:, 0] * scale, sc_rel_earth[:, 1] * scale, 'b-', label='Spacecraft')
ax.plot(moon_rel_earth[:, 0] * scale, moon_rel_earth[:, 1] * scale, 'gray', linestyle='--', label='Moon orbit')
ax.scatter([0], [0], c='blue', s=200, label='Earth')
ax.scatter([moon_rel_earth[-1, 0] * scale], [moon_rel_earth[-1, 1] * scale], c='gray', s=100, label='Moon (final)')
ax.set_xlabel('X (1000 km)')
ax.set_ylabel('Y (1000 km)')
ax.set_title('Earth-Moon Transfer (XY plane)')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Distance from Earth over time
ax = axes[1]
distances = np.linalg.norm(sc_rel_earth, axis=1) * scale
ax.plot(result.t, distances, 'b-', linewidth=2)
ax.axhline(y=384.4, color='gray', linestyle='--', label='Moon distance')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Distance from Earth (1000 km)')
ax.set_title('Distance from Earth')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Maximum distance from Earth: {distances.max():.0f} thousand km")

## 6. Mission Summary

In [ ]:
# Final state analysis
final_r = result.y[:3, -1]
final_v = result.y[3:6, -1]
final_m = result.y[6, -1]

# Distance from Moon at end
r_moon_final = get_body_position('moon', result.t[-1])
d_moon_km = np.linalg.norm(final_r - r_moon_final) * 1.496e8

# Velocity relative to Moon
v_moon = get_body_velocity('moon', result.t[-1])
v_rel_moon = final_v - v_moon
v_rel_moon_kmps = np.linalg.norm(v_rel_moon) * 1731.46

print("=" * 50)
print("MISSION SUMMARY")
print("=" * 50)
print(f"Total flight time: {result.t[-1]:.2f} days")
print(f"Initial mass: {state[6]:,.0f} kg")
print(f"Final mass: {final_m:,.0f} kg")
print(f"Propellant consumed: {state[6] - final_m:,.0f} kg")
print(f"")
print(f"Distance from Moon: {d_moon_km:,.0f} km")
print(f"Velocity relative to Moon: {v_rel_moon_kmps:.2f} km/s")
print(f"")
print(f"Note: LOI burn of ~{v_rel_moon_kmps - 1.0:.1f} km/s needed for lunar orbit capture")